# StrikeTwice and the Task 2 benchmark

StrikeTwice applies EVT once to the graph and once to each path's ordered occurrences. The first CPU example shows two paths sharing a graph encoding. The Task 2 benchmark then evaluates a trained model on held-out graphs.

## Build a batch with two queries

The first path revisits vertex 0. The second has a padded final position, represented by `-1` and masked by `path_mask`. Both rows have identical graph features and vertex masks, so they receive the same `graph_index`.

In [ ]:
import torch
from path_augmented_transformer import GraphPathBatch, ScalarPathHead, StrikeTwice

# Build one directed graph with a vertex feature and an adjacency channel.
torch.manual_seed(7)
X = torch.tensor([[[1.0], [2.0], [3.0], [4.0]]])
Y = torch.zeros(1, 4, 4, 1)
for source, target in [(0, 1), (1, 0), (0, 2), (2, 3)]:
    Y[0, source, target, 0] = 1.0

# Use the same graph ID for both queries to share the first EVT encoding.
p = torch.tensor([[0, 1, 0, 2], [0, 2, 3, -1]], dtype=torch.long)
num_queries = p.shape[0]
batch = GraphPathBatch(
    X=X.expand(num_queries, -1, -1),
    Y=Y.expand(num_queries, -1, -1, -1),
    p=p,
    node_mask=torch.ones(num_queries, 4, dtype=torch.bool),
    path_mask=p >= 0,
    graph_index=torch.zeros(num_queries, dtype=torch.long),
)

## Encode the paths and attach a prediction head

`out.X` and `out.Y` are the first EVT's graph states. `out.Z` contains one state per path occurrence, and `out.R` contains one state per ordered occurrence pair. `ScalarPathHead` pools the valid states in `out.Z` to produce one value per path. The model and head use randomly initialized weights.

In [ ]:
# Use one EVT layer per stage and a scalar head over path occurrences.
model = StrikeTwice(
    d_x=1,
    d_y=1,
    d_h=8,
    max_path_length=4,
    graph_num_layers=1,
    path_num_layers=1,
    num_heads=2,
).eval()
head = ScalarPathHead(d_h=8).eval()

# Encode the graph, then process each query in its path order.
with torch.inference_mode():
    out = model(batch)
    predictions = head(out, batch.path_mask).prediction

# Report the occurrence shapes and the prediction for each unpadded path.
print(f'Path vertex states {tuple(out.Z.shape)}')
print(f'Path pair states {tuple(out.R.shape)}')
for path, mask, prediction in zip(batch.p, batch.path_mask, predictions):
    print(f'Path {path[mask].tolist()}  prediction {prediction.item():.6f}')

## Verify the shared encoding

With `graph_index=None`, StrikeTwice encodes each graph row independently. In evaluation mode this yields the same path states and predictions as the shared encoding, up to floating-point rounding.

In [ ]:
from dataclasses import replace

# Recompute the graph encoding independently for each query.
with torch.inference_mode():
    independent = model(replace(batch, graph_index=None))
    independent_predictions = head(independent, batch.path_mask).prediction

# Compare both occurrence streams and the final predictions.
torch.testing.assert_close(out.Z, independent.Z)
torch.testing.assert_close(out.R, independent.R)
torch.testing.assert_close(predictions, independent_predictions)
print('Shared and independent graph encodings give the same predictions.')

## Task 2 benchmark

Task 2 predicts alternative-route counts for a supplied path in a layered directed acyclic graph (DAG). The target is the mean of the natural logarithms of the directed route counts between pairs of path positions at least two steps apart. The counts use the full graph, including vertices outside the query path.

The release includes the canonical held-out split of 256 graphs and 1,024 queries, and the validation-selected `ST[EVT, EVT]` checkpoint for seed 0 (`final_tt_lr3em04_s0`, step 1,000 of 2,000). Each graph has seven layers of four vertices and four native seven-vertex queries. The packaged fixture contains the original graph features, raw and standardized targets, archived predictions, and source SHA-256 digests.

The model and cache benchmark use CUDA when available, then Apple Metal (MPS), then CPU. Set `STRIKETWICE_TASK2_DEVICE` to `cuda`, `mps`, or `cpu` before starting Jupyter to select a backend explicitly. The exact integer route counts and double-precision scores are calculated on CPU.

In [ ]:
from gzip import decompress
from importlib.resources import files
from io import BytesIO
from os import getenv

import torch
from torch import nn
from path_augmented_transformer import GraphPathBatch, PathEncoderOutput, ScalarPathHead, StrikeTwice

# Resolve the requested backend and keep inference in full FP32.
available_devices = {
    'cuda': torch.cuda.is_available(),
    'mps': torch.backends.mps.is_available(),
    'cpu': True,
}
requested_device = getenv('STRIKETWICE_TASK2_DEVICE', 'auto')
if requested_device == 'auto':
    requested_device = next(name for name, available in available_devices.items() if available)
if requested_device not in available_devices:
    raise ValueError('Select auto, cuda, mps, or cpu for STRIKETWICE_TASK2_DEVICE.')
if not available_devices[requested_device]:
    raise RuntimeError(f'PyTorch cannot access the requested {requested_device} device.')
task2_device = torch.device(requested_device)
if task2_device.type == 'cpu':
    torch.set_num_threads(4)
torch.set_float32_matmul_precision('highest')
print(f'Task 2 inference device {task2_device} (FP32)')

# Load the held-out inputs, checkpoint and archived predictions on CPU.
resource = files('path_augmented_transformer').joinpath('resources').joinpath('task2_reference.pt.gz')
task2 = torch.load(BytesIO(decompress(resource.read_bytes())), map_location='cpu', weights_only=True)

# Sum adjacency powers to count routes of up to six edges in the DAG.
path = task2['paths'][0, 0]
adjacency = (task2['Y'][0, ..., 0] > 0.5).to(torch.int64)
power = adjacency
route_counts = adjacency.clone()
for _ in range(2, len(path)):
    power = power @ adjacency
    route_counts += power
# Average log route counts across nonconsecutive path positions.
start, end = torch.triu_indices(len(path), len(path), offset=2)
raw_score = route_counts[path[start], path[end]].double().log().mean()
torch.testing.assert_close(raw_score, task2['raw_target'][0, 0].double(), atol=2e-7, rtol=0)
print(f'Exact raw target for the first test query {raw_score.item():.6f}')

### Evaluate the trained checkpoint

Load the two-EVT encoder and scalar head, and group the four queries per graph with `graph_index`. The checkpoint predicts standardized targets using the paper's training-pool normalization. The benchmark reports MSE on those targets and test R², and checks every prediction against the archive.

In [ ]:
# Recreate the paper's two EVT stacks and restore the selected weights.
task2_model = nn.ModuleDict({
    'encoder': StrikeTwice(
        d_x=8,
        d_y=4,
        d_h=32,
        max_path_length=7,
        graph_num_layers=2,
        path_num_layers=2,
        num_heads=4,
        graph_triangular_head_config='balanced',
        path_triangular_head_config='balanced',
    ),
    'head': ScalarPathHead(d_h=32),
})
task2_model.load_state_dict(task2['state_dict'])
task2_model.to(task2_device).eval()
task2_encoder = task2_model['encoder']
task2_head = task2_model['head']


# Assemble graph and path inputs before moving the batch to the device.
def make_task2_batch(graph_rows, paths):
    X = task2['X'][graph_rows]
    return GraphPathBatch(
        X=X,
        Y=task2['Y'][graph_rows],
        p=paths,
        node_mask=torch.ones(X.shape[:2], dtype=torch.bool),
        path_mask=torch.ones_like(paths, dtype=torch.bool),
        graph_index=graph_rows,
    ).to(task2_device)


# Evaluate in chunks so the queries of each graph share its encoding.
with torch.inference_mode():
    chunks = []
    for graph_rows in torch.arange(len(task2['X'])).split(8):
        paths = task2['paths'][graph_rows]
        queries = make_task2_batch(graph_rows.repeat_interleave(paths.shape[1]), paths.flatten(0, 1))
        encoded = task2_encoder(queries)
        chunks.append(task2_head(encoded, queries.path_mask).prediction.ravel())
    prediction = torch.cat(chunks).cpu().double()

# Check against the archive and compute regression scores in float64.
torch.testing.assert_close(prediction, task2['saved_prediction'].ravel(), atol=1e-5, rtol=1e-5)
target = task2['target'].ravel()
squared_error = (prediction - target).square()
r_squared = 1 - squared_error.sum() / (target - target.mean()).square().sum()
print(f'Test predictions verified against the archive ({prediction.numel()} queries)')
print(f'Test MSE {squared_error.mean().item():.6f}  |  Test R² {r_squared.item():.6f}')

For `ST[EVT, EVT]`, the paper reports a Task 2 test R² of **0.589 ± 0.102** (mean ± sample standard deviation over five paired seeds). The seed-0 checkpoint evaluated above reaches **0.597** on the same test split.

### Benchmark graph-cache reuse

Time the two-EVT checkpoint on the paper's 12 timing graphs with either 4 or 16 queries per graph, processed one at a time. Four queries use the original test paths. The 16-query request adds 12 distinct paths from the paper's timing list.

Both schedules start with an empty cache and process identical paths with the same weights. Rebuild encodes the graph for each path. Reuse encodes it once per request. Each measured request includes initial graph encoding, occurrence selection, the second EVT, and the prediction head. Batch assembly and input transfers happen before timing. On CUDA and MPS, the timer waits for each request to finish on the device.

The notebook checks matching outputs in one warm-up per schedule, then averages graph-level medians of three timed repetitions. Timings depend on the device. The paper's CPU measurements over five seeds, with seven timed repetitions per graph, gave speedups of 2.40× for four queries and 3.78× for sixteen.

In [ ]:
from statistics import mean, median
from time import perf_counter


@torch.inference_mode()
def run_task2_request(batches, reuse):
    # Start with an empty cache for every measured request.
    cache = None
    predictions = []
    for batch in batches:
        if cache is None or not reuse:
            cache = task2_encoder.graph_backbone(batch.X, batch.Y, batch.node_mask)
        # Select states in occurrence order, then run the second EVT and head.
        selected = task2_encoder.select_occurrence_graph(cache, batch)
        occurrences = task2_encoder.path_backbone(selected.X, selected.Y, batch.path_mask)
        encoded = PathEncoderOutput(X=cache.X, Y=cache.Y, Z=occurrences.X, R=occurrences.Y)
        predictions.append(task2_head(encoded, batch.path_mask).prediction)
    return torch.cat(predictions)


def synchronize_task2_device():
    # Wait for queued GPU operations before reading the wall clock.
    if task2_device.type == 'cuda':
        torch.cuda.synchronize(task2_device)
    elif task2_device.type == 'mps':
        torch.mps.synchronize()


def benchmark_task2(query_count, repeats=3):
    graph_medians = {False: [], True: []}
    for row, paths in zip(task2['timing_rows'], task2['timing_paths']):
        # Prepare inputs and check both schedules during warm-up.
        batches = [make_task2_batch(row.unsqueeze(0), path) for path in paths[:query_count].split(1)]
        rebuilt = run_task2_request(batches, reuse=False)
        cached = run_task2_request(batches, reuse=True)
        torch.testing.assert_close(rebuilt.cpu(), cached.cpu(), atol=1e-5, rtol=1e-5)

        # Time complete requests after input assembly and device transfers.
        samples = {False: [], True: []}
        for _ in range(repeats):
            for reuse in (False, True):
                synchronize_task2_device()
                start = perf_counter()
                run_task2_request(batches, reuse=reuse)
                synchronize_task2_device()
                samples[reuse].append(1000 * (perf_counter() - start))
        for reuse in samples:
            graph_medians[reuse].append(median(samples[reuse]))

    # Give each timing graph the same weight by averaging its median.
    rebuild_ms, reuse_ms = mean(graph_medians[False]), mean(graph_medians[True])
    print(f'{query_count:2d} queries | {rebuild_ms:7.2f} ms | {reuse_ms:7.2f} ms | {rebuild_ms / reuse_ms:.2f}x')


print('Request    |   Rebuild  |     Reuse  | Speedup')
benchmark_task2(4)
benchmark_task2(16)